# LeetCode 28 — Find the Index of the First Occurrence in a String | Nhập môn Python, giảng từng bước

**Độ khó:** Easy · **Kỹ thuật trọng tâm:** String · Brute Force · đối chiếu ký tự  
**Bài gốc:** https://leetcode.com/problems/find-the-index-of-the-first-occurrence-in-a-string/

Notebook này dành cho người **mới bắt đầu**. Đọc các ô từ trên xuống: trước hết hiểu **vì sao**, sau đó chạy code nhỏ để tự thấy **cách làm**.

**Khung hàm (signature) cần nộp LeetCode:**

```python
class Solution:
    def strStr(self, haystack: str, needle: str) -> int:
        ...
```

**Sau bài học, bạn có thể:**
- Phân biệt substring với prefix; hiểu chỉ số bắt đầu từ 0.
- Tự triển khai dò từng điểm bắt đầu mà không dùng hàm tìm kiếm có sẵn.
- Giải thích đúng O((n-m+1)·m) worst-case, test đầu/cuối/không tìm thấy.

**Cách sử dụng:** chạy từng code cell theo thứ tự bằng `Shift + Enter`. Những ô có `TODO` là bài tự luyện: chỉ định nghĩa hàm hoặc chứa lời nhắc kiểm thử, **không được gọi tự động** khi chưa làm xong.

> Lộ trình: hiểu đề → Python căn bản → cách trực giác → phát hiện nút thắt → cải tiến → trace → nộp LeetCode → test → lỗi → vận dụng → bài tập → đáp án → checklist.

## 1. Bài toán đang yêu cầu gì?

Cho chuỗi lớn `haystack` và chuỗi nhỏ `needle`. Hãy trả **chỉ số xuất hiện đầu tiên** của `needle` trong `haystack`. Nếu không xuất hiện, trả `-1`.

| haystack | needle | trả về | vì sao |
|---|---|---:|---|
| `"sadbutsad"` | `"sad"` | `0` | thấy ở vị trí 0 và 6; cần đầu tiên |
| `"leetcode"` | `"leeto"` | `-1` | không xuất hiện |
| `"bananana"` | `"ana"` | `1` | bắt đầu ở chỉ số 1 trước chỉ số 3, 5 |

`substring` (chuỗi con) có thể nằm ở **giữa** chuỗi, không nhất thiết ở đầu. `haystack` và `needle` chỉ là tên biến theo đề.

In [ ]:
haystack = "bananana"; needle = "ana"
print("vị trí 0:", haystack[0:3]); print("vị trí 1:", haystack[1:4])
print("cách Python tự tìm:", haystack.find(needle))

## 2. Python cơ bản: chỉ số bắt đầu bằng 0

`s[i]` lấy **một ký tự** ở vị trí `i`; `len(s)` cho độ dài. Chẳng hạn `"cat"` có các vị trí 0, 1, 2. **Không** được đọc `s[len(s)]` vì vị trí này nằm ngoài chuỗi.

In [ ]:
s="cat"
for i in range(len(s)):
    print("i=",i,"ký tự=",repr(s[i]))
print("len=",len(s))

## 3. Cắt chuỗi `s[start:end]` có nghĩa gì?

Dạng **slicing** `s[start:end]` lấy các vị trí từ `start` đến **trước** `end`. Ví dụ: `"banana"[1:4] == "ana"` (lấy 1,2,3). Nếu `m = len(needle)`, đoạn bắt đầu tại `i` là `haystack[i:i+m]`.

Lưu ý: tạo slice có thể tạo **chuỗi mới** dài tối đa `m`; về sau ta sẽ có cách đối chiếu từng ký tự không cần slice.

In [ ]:
s="banana"; m=3
for i in range(4):
    print(f"s[{i}:{i+m}] =",repr(s[i:i+m]))

## 4. Trực giác: trượt một cửa sổ dài m

Hãy tưởng tượng đặt một khung dài `m` lên `haystack`, bắt đầu từ trái sang phải:

`haystack = "cabcab"`, `needle = "cab"`, `m=3`.

- `i=0`: đoạn `"cab"` bằng needle → trả `0` ngay.
- Nếu chưa khớp, dịch khung sang vị trí `i=1`, rồi `i=2`, ...

Chỉ có thể bắt đầu tại `i <= n-m`: nếu bắt đầu muộn hơn, không còn đủ `m` ký tự để so sánh.

In [ ]:
n=6; m=3
print("n-m =",n-m,"; các vị trí thử:",list(range(n-m+1)))
print("Nếu n=2, m=3 ->",list(range(2-3+1)),"(không có vị trí hợp lệ)")

## 5. Lời giải trực giác dùng slicing

Duyệt các vị trí `i` hợp lệ. Tạo `haystack[i:i+m]` và so với `needle`; gặp bằng nhau thì trả ngay. Nếu duyệt hết vẫn chưa thấy, trả `-1`.

Mặt hạn chế: mỗi lượt cắt có thể tạo thêm chuỗi tạm kích thước `m`.

In [ ]:
def find_with_slices(haystack: str, needle: str) -> int:
    n,m=len(haystack),len(needle)
    for i in range(n-m+1):
        if haystack[i:i+m] == needle:
            return i
    return -1

print(find_with_slices("sadbutsad","sad"))
print(find_with_slices("leetcode","leeto"))

## 6. Cải tiến: chỉ so ký tự, không tạo lát cắt

Ta có thể kiểm tra các cặp `(haystack[i+j], needle[j])`, với `j=0..m-1`. Nếu thấy khác, ngừng so tại vị trí `i` và chuyển sang `i+1`. Nếu đủ `m` cặp đều giống, trả `i`.

`break`: thoát **vòng lặp trong** ngay; `for ... else`: nhánh `else` chỉ chạy khi **không có break**. Để dễ hiểu, phần code chính dùng một biến `matched` thay cho `for...else`.

In [ ]:
haystack="banana"; needle="ana"; i=1
for j in range(len(needle)):
    print(f"j={j}: haystack[{i+j}]={haystack[i+j]!r}, needle[{j}]={needle[j]!r}")

## 7. Code chính: so từng ký tự

Vòng ngoài đi qua mọi vị trí bắt đầu hợp lệ; vòng trong đi qua mỗi ký tự của `needle`. Biến `matched=True` ban đầu, đổi thành False và `break` khi gặp cặp khác. Cách này **không tạo chuỗi con tạm**.

In [ ]:
def find_first_manual(haystack: str, needle: str) -> int:
    n,m=len(haystack),len(needle)
    for i in range(n-m+1):
        matched=True
        for j in range(m):
            if haystack[i+j] != needle[j]:
                matched=False
                break
        if matched:
            return i
    return -1

for h,nd in [("sadbutsad","sad"),("leetcode","leeto"),("aaaaa","bba")]:
    print(h,nd,"->",find_first_manual(h,nd))

## 8. Bảng trace bằng tay

Tìm `"abc"` trong `"ababc"`:

| i | Cặp ký tự so theo thứ tự | Kết luận |
|---:|---|---|
| 0 | a=a, b=b, a≠c | sai tại j=2, sang i=1 |
| 1 | b≠a | sai ngay, sang i=2 |
| 2 | a=a, b=b, c=c | đủ 3 ký tự → trả 2 |

**Bất biến:** trước khi thử vị trí `i`, ta đã kiểm tra và loại bỏ **mọi** vị trí nhỏ hơn `i`.

In [ ]:
def trace_find(haystack,needle):
    n,m=len(haystack),len(needle)
    for i in range(n-m+1):
        print(f"\nThử i={i}")
        ok=True
        for j in range(m):
            same=haystack[i+j]==needle[j]
            print(f"  j={j}: {haystack[i+j]!r} với {needle[j]!r} -> {same}")
            if not same:
                ok=False; break
        if ok:
            print("KHỚP, trả",i); return i
    print("KHÔNG CÓ, trả -1"); return -1

assert trace_find("ababc","abc")==2

## 9. Vì sao thuật toán đúng?

Để `needle` xuất hiện từ `i`, tất cả `m` ký tự tương ứng phải bằng nhau. Vòng trong kiểm tra **đúng điều kiện ấy**, nên không báo nhầm. Vòng ngoài thử `i` theo thứ tự từ 0 tăng dần và trả ngay kết quả đầu tiên, nên vị trí trả về **nhỏ nhất**. Không gặp vị trí nào thì bắt buộc trả -1.

## 10. Độ phức tạp và các giới hạn

Gọi `n=len(haystack)`, `m=len(needle)`. Có tối đa `max(n-m+1,0)` vị trí bắt đầu và tại mỗi vị trí so tối đa `m` ký tự:

- **Time:** O((n-m+1)·m) trong trường hợp `n>=m`, tệ nhất O(n·m). Ví dụ `"aaaaaaaa...ab"` so với `"aaaab"` dễ gây so sánh lặp.
- **Space phụ:** O(1) vì chỉ dùng chỉ số và cờ Boolean.
- `str.find` là hàm có sẵn trong Python, rất tiện khi làm việc thực tế, nhưng chưa thay thế được tư duy tự giải thuật toán tìm chuỗi.

In [ ]:
print("banana".find("ana"))  # 1
print("banana".find("xyz"))  # -1
print("aaaba".find("aab"))   # 1

## 11. Code nộp LeetCode

LeetCode yêu cầu tên hàm **`strStr`** và hai tham số `haystack`, `needle`; kết quả là **một số nguyên**. Copy riêng class dưới đây, không đem theo các lệnh `print` hay `trace_*`.

In [ ]:
class Solution:
    def strStr(self, haystack: str, needle: str) -> int:
        n,m=len(haystack),len(needle)
        for i in range(n-m+1):
            for j in range(m):
                if haystack[i+j]!=needle[j]:
                    break
            else:
                return i
        return -1

print(Solution().strStr("sadbutsad","sad"))

## 12. Giải thích cú pháp `for ... else` trong code submit

`else` thụt vào ngang hàng với `for j`; Python chạy `else` nếu vòng `for` **kết thúc mà không có `break`**. Ở đây điều đó tương đương mọi cặp ký tự đều trùng. Nếu cú pháp này còn lạ, bạn có thể nộp bản dùng `matched` ở phần 7 — cả hai đều đúng.

In [ ]:
for x in [1,2,3]:
    if x==2: break
else:
    print("Không in vì có break")
for x in [1,2,3]:
    if x==9: break
else:
    print("Có in vì không có break")

## 13. Kiểm thử mẫu và edge cases

Test vị trí 0, giữa, cuối, không thấy, chuỗi bằng nhau, `needle` dài hơn `haystack` và nhiều lần xuất hiện. Theo đề, `needle` không rỗng; hỗ trợ rỗng ở hàm demo là mở rộng hợp lý với kết quả 0.

In [ ]:
cases=[("sadbutsad","sad",0),("leetcode","leeto",-1),("banana","ana",1),
       ("abc","abc",0),("abc","abcd",-1),("abc","c",2),
       ("aaaa","aa",0),("abc","d",-1),("a","a",0)]
for h,nd,want in cases:
    got=Solution().strStr(h,nd)
    assert got==want,(h,nd,got,want)
    assert find_first_manual(h,nd)==want
print("PASS",len(cases),"trường hợp")

## 14. Lỗi thường gặp: sai phạm vi và trả nhầm

❌ `range(n-m)` **bỏ qua vị trí cuối** hợp lệ `n-m`; ví dụ `"abc"` tìm `"c"` phải thử `i=2`.

❌ `return -1` **bên trong** vòng ngoài sẽ dừng ngay sau lần thử đầu, bỏ qua kết quả về sau.

❌ `return haystack[i]` trả ký tự, trong khi đề muốn **chỉ số nguyên** `i`.

✅ `range(n-m+1)`, chỉ `return -1` sau khi vòng ngoài chạy xong.

In [ ]:
n=3;m=1
print("SAI:",list(range(n-m)))
print("ĐÚNG:",list(range(n-m+1)))
assert Solution().strStr("abc","c")==2

## 15. Khi nào nhận ra pattern này?

Khi cần tìm một mẫu **liền nhau** trong chuỗi, hãy thử từng vị trí bắt đầu (string matching). Với dữ liệu lớn hoặc rất nhiều phép tìm, học thêm KMP/Z-algorithm để tối ưu trường hợp xấu xuống O(n+m). **Không bắt buộc** phải biết KMP mới giải được bài Easy này.

## 16. Tự luyện 5 bài (chưa tự chạy TODO)

### Bài 1 — chỉ số và slicing — Tự làm

Trả đoạn con dài 3 ký tự bắt đầu ở vị trí i.

**Gợi ý cấp 1 (tư duy):** Tập trung vào `s[start:end]`.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

Độ dài lấy bằng end-start.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Dùng `s[i:i+3]`.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_28_1(s: str, i: int) -> str:
    # TODO: lấy 3 ký tự
    pass

# assert exercise_28_1("abcdef",2)=="cde"

### Bài 2 — tính tay — Tự làm

Trả vị trí đầu của `"ab"` trong `"xxababa"`.

**Gợi ý cấp 1 (tư duy):** Thử i từ 0.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

Đoạn cần tìm dài 2.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Hai ký tự ở i=2 là a,b.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_28_2() -> int:
    # TODO: tự đi từng vị trí
    pass

# assert exercise_28_2()==2

### Bài 3 — code chính — Tự làm

Viết lại cách dùng slicing nhưng không gọi `.find()`.

**Gợi ý cấp 1 (tư duy):** Mỗi cửa sổ dài m.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

Dùng `range(n-m+1)`.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Nếu slice bằng needle, trả i; hết vòng trả -1.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_28_3(haystack: str, needle: str) -> int:
    # TODO: duyệt i=0..n-m và so sánh slice
    pass

# assert exercise_28_3("abcde","cd")==2

### Bài 4 — đếm vị trí xuất hiện — Tự làm

Biến thể: đếm tất cả vị trí bắt đầu mà needle xuất hiện, **kể cả chồng lấp**.

**Gợi ý cấp 1 (tư duy):** Thử đầy đủ mọi i.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

Khởi tạo count=0.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Mỗi lần khớp `count += 1`.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_28_4(haystack: str, needle: str) -> int:
    # TODO: đếm thay vì return ngay
    pass

# assert exercise_28_4("aaaa","aa")==3

### Bài 5 — tự viết so ký tự — Tự làm

Viết lại lời giải chính không slicing và không `.find`.

**Gợi ý cấp 1 (tư duy):** Mỗi j so một cặp ký tự.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

Có thể dùng biến matched.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Khác ký tự thì break; đủ m ký tự thì return i.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_28_5(haystack: str, needle: str) -> int:
    # TODO: vòng ngoài i, vòng trong j
    pass

# assert exercise_28_5("mississippi","issip")==4

## 17. Đáp án tham khảo — chỉ mở sau khi tự làm

In [ ]:
def answer_28_1(s,i): return s[i:i+3]
def answer_28_2(): return 2
def answer_28_3(h,nd):
    for i in range(len(h)-len(nd)+1):
        if h[i:i+len(nd)]==nd:return i
    return -1
def answer_28_4(h,nd):
    count=0
    for i in range(len(h)-len(nd)+1):
        if h[i:i+len(nd)]==nd: count+=1
    return count
def answer_28_5(h,nd):
    for i in range(len(h)-len(nd)+1):
        matched=True
        for j in range(len(nd)):
            if h[i+j]!=nd[j]: matched=False; break
        if matched:return i
    return -1
assert answer_28_1("abcdef",2)=="cde" and answer_28_2()==2
assert answer_28_3("abcde","cd")==2 and answer_28_4("aaaa","aa")==3
assert answer_28_5("mississippi","issip")==4
print("Đáp án: PASS 5/5")

## Tóm tắt — mẫu tư duy cần nhớ

**Mẫu:** xét lần lượt mọi điểm bắt đầu `i=0..n-m`, so `needle[j]` với `haystack[i+j]`; nếu mọi ký tự khớp thì trả `i`, không có thì -1. **Worst-case:** O(n·m), **Space:** O(1).

## Checklist trước khi submit

- [ ] Đọc đúng index 0-based và phân biệt prefix/substring.
- [ ] Không quên i=n-m và return -1 chỉ sau vòng ngoài.
- [ ] Tự dựng trace với "ababc"/"abc".
- [ ] Test khớp đầu, giữa, cuối, không khớp.
- [ ] Có thể viết lại code không xem đáp án.

**Tự kiểm tra lần cuối:** có thể giải thích lời giải bằng lời trước khi xem code? Có tự tính được ít nhất một bảng trace? Nếu chưa, quay lại phần mô phỏng.

*Notebook học tập được biên soạn theo cẩm nang tiếng Việt đính kèm; tiêu đề/đề/ràng buộc bám sát LeetCode. Những biến thể được đánh dấu là mở rộng, không phải yêu cầu đề chính.*